# P14 — 圆周取样点数：噪声底与真实误差是**反相关**的

**2026-10-01.** 本 notebook 记录 altharm 的第六个盲区，并把每一步的检验都跑一遍。

altharm 的核心方法是：在 k 空间的圆周上把自旋劈裂 ΔE(k) 展成 Σ_l [a_l cos(lθ) + b_l sin(lθ)]，
**对称性禁戒的 l，其拟合幅度就是数值误差**（"噪声底"）。

P3/P12/P13 已经发现这个误差估计对五个输入是**盲**的（U 每 eV 21%、展宽 12.9%、
PAW vs USPP 9.7%、PBEsol vs PBE 7.3%、几何 <4%，而噪声底自己只报 0.04%）。
盲是可以接受的——一个注意不到某类误差的数，对它能注意到的误差仍然没说错。

**这一节发现的第六个输入比盲更糟：圆周上的取样点数 N 同时决定噪声底和真实误差，而且方向相反。**

## 0. 环境与数据

In [1]:
import sys, re, json
from math import gcd
from pathlib import Path
import numpy as np

ROOT = Path('/scratch/work/yuc11/altharm')
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT/'tests'))
np.set_printoptions(suppress=True)

RUNS = Path('/scratch/work/yuc11/altharm_runs')
XY   = ROOT/'tests/data/harmwave_v2/harmwave_report_v2.txt'   # k_x k_y 圆周, 152 点, lmax=45
XY_K = ROOT/'tests/data/harmwave_v2/kcirc_xy_tpiba.npy'
YZ   = ROOT/'tests/data/harmwave_yz/harmwave_yz.txt'          # k_y k_z 大圆, 152 点, lmax=24
YZ_K = ROOT/'tests/data/harmwave_yz/kcirc_tpiba.npy'
for p in (XY, XY_K, YZ, YZ_K):
    print(f"{'✓' if p.exists() else '✗'}  {p.relative_to(ROOT)}")

✓  tests/data/harmwave_v2/harmwave_report_v2.txt
✓  tests/data/harmwave_v2/kcirc_xy_tpiba.npy
✓  tests/data/harmwave_yz/harmwave_yz.txt
✓  tests/data/harmwave_yz/kcirc_tpiba.npy


两条圆周都来自**同一个**已收敛的 Cr₂SeAs 密度（24×24×8，degauss 0.0150，U=3.0，带 44），
只是平面不同：

| | 平面 | 法向 | 偏移 | 点数 |
|---|---|---|---|---|
| `harmwave_v2` | k_x k_y | (0,0,1) | k_z = 0.25 b₃ | 152 |
| `harmwave_yz` | k_y k_z | (1,0,0) | 0（过原点的大圆） | 152 |

k_y k_z 这条是 2026-09-22 的作业 20383535，用来验证 harmwave.x v2 "从 k 点读平面"
那段代码在非 +ẑ 平面上也对。它跑完了但当时没来得及看结果。

## 1. 先看 y-z 圆周：v2 在任意平面上对不对

In [2]:
def parse(path):
    ops, coef, theta, dE, hdr = [], {}, [], [], {}
    for line in Path(path).read_text().splitlines():
        t = line.split()
        if   line.startswith('OP '): ops.append((int(t[1]), np.array([float(x) for x in t[2:11]]).reshape(3,3)))
        elif line.startswith('CF '): coef[(int(t[1]), t[2])] = float(t[3])
        elif line.startswith('PT '): theta.append(float(t[1])); dE.append(float(t[2]))
        elif line.startswith('#') and '=' in line:
            k, v = line[1:].split('=', 1); hdr[k.strip()] = v.strip()
    return ops, coef, np.array(theta), np.array(dE), hdr

ops_yz, cf_yz, th_yz, dE_yz, hdr_yz = parse(YZ)
for k in ('plane normal (cartesian, 2pi/alat)', 'plane offset  n.k', 'theta frame e1',
          'theta frame e2', 'nsym (crystallographic, before plane filter)',
          'nkept (plane-preserving, chi-classified ops)'):
    print(f"  {k:<46} = {hdr_yz[k]}")

  plane normal (cartesian, 2pi/alat)             = 1.000000000     0.000000000     0.000000000
  plane offset  n.k                              = 0.000000000
  theta frame e1                                 = 0.000000000     1.000000000     0.000000000
  theta frame e2                                 = 0.000000000     0.000000000     1.000000000
  nsym (crystallographic, before plane filter)   = 12
  nkept (plane-preserving, chi-classified ops)   = 4


法向读成 (1,0,0)、偏移 0、框架 e1=ŷ e2=ẑ —— **harmwave.x 确实是从 k 点读的平面，没有退回 +ẑ 的假设。**
12 个晶体学操作里有 4 个保留这条圆周。

接下来看系数。预言（独立地由 `altharm.symmetry` 给出）是 **允许 l = 2,4,6,…,24**，全是偶数。

In [3]:
odd  = np.array([abs(cf_yz[(l,k)]) for l in range(1,25,2) for k in 'cs'])
ec   = np.array([abs(cf_yz[(l,'c')]) for l in range(2,25,2)])
es   = np.array([abs(cf_yz[(l,'s')]) for l in range(2,25,2)])
print(f"  奇数 l (cos 和 sin 都算), n={len(odd):2d} : 最大 {odd.max():.3e} eV")
print(f"  偶数 l 的 cos 通道,       n={len(ec):2d} : 最大 {ec.max():.3e} eV")
print(f"  偶数 l 的 sin 通道,       n={len(es):2d} : 最大 {es.max():.3e} eV   <-- 信号")
print(f"\n  ΔE(θ=0)，也就是 k_z=0 处 = {dE_yz[0]*1e6:+.3f} μeV  (理应为 0)")

  奇数 l (cos 和 sin 都算), n=24 : 最大 1.000e-14 eV
  偶数 l 的 cos 通道,       n=12 : 最大 5.724e-07 eV
  偶数 l 的 sin 通道,       n=12 : 最大 2.036e-02 eV   <-- 信号

  ΔE(θ=0)，也就是 k_z=0 处 = -3.405 μeV  (理应为 0)


**比预言还要尖锐。** 不只是"偶数 l 允许"，而是**偶数 l 且只有 sin**：

- 奇数 l 精确为 0（14 位小数全零）
- 偶数 l 的 cos ~1e-7 eV
- 偶数 l 的 sin 最大 20 meV

这三条正好是那 4 个保留操作的内容：C2x（χ=+1，θ→θ+π）强制偶数；
M_z（χ=−1，θ→−θ）强制奇函数即纯 sin；M_y（χ=−1，θ→π−θ）与前两条相容。
纯 sin 等价于 **ΔE 关于 k_z 是奇的**，而 ΔE(k_z=0) = −3.4 μeV ≈ 0 正是这件事的直接体现。

## 2. 可是那些"精确的零"是怎么来的？

14 位全零让人不安：**如果 harmwave.x 在拟合前先对数据做了对称化，禁戒谐波就会按构造为零，
噪声底也就无从读起**——那会动摇整个方法的根基。先去 Fortran 源码里确认。

In [4]:
src = (ROOT/'qe_native/harmwave.f90').read_text().splitlines()
for n, line in enumerate(src, 1):
    if re.search(r'design\(:|DGELSS', line):
        print(f"  harmwave.f90:{n}: {line.strip()}")

  harmwave.f90:82: REAL(DP), ALLOCATABLE :: theta(:), dE(:), design(:,:), coef(:), work(:)
  harmwave.f90:286: design(:,1) = 1.0_DP
  harmwave.f90:288: design(:,2*l)   = COS( l * theta(:) )
  harmwave.f90:289: design(:,2*l+1) = SIN( l * theta(:) )
  harmwave.f90:295: CALL DGELSS( n, ncoef, 1, design, n, coef, MAX(n,ncoef), s_sv, -1.0_DP, &
  harmwave.f90:297: IF ( info /= 0 ) CALL errore( 'harmwave', 'DGELSS failed', info )


设计矩阵就是 1、cos(lθ)、sin(lθ) 三类原始列，然后一个 `DGELSS` 最小二乘。
**没有任何对称化。** 所以那些零是**数据本身**的性质，不是拟合的产物。

那就值得直接问数据：每个对称操作，ΔE(Rk) = χ·ΔE(k) 这条约束实际被满足到什么程度？

## 3. 检验一：χ=+1 与 χ=−1 是两个完全不同量级的通道

In [5]:
def plane_action(R, k, e1, e2):
    """R 在圆周上的作用 θ -> a·θ + b，以及它是否把样点置换成样点。"""
    kr  = k @ R.T
    th0 = np.arctan2(k @ e2,  k @ e1)
    th1 = np.arctan2(kr @ e2, kr @ e1)
    cand = []
    for a in (1, -1):
        b = float(np.angle(np.mean(np.exp(1j*(th1 - a*th0)))))      # 圆周均值, b=π 也安全
        err = np.abs(np.angle(np.exp(1j*(th1 - (a*th0 + b))))).max()
        cand.append((err, a, b))
    err, a, b = min(cand)
    perm = np.linalg.norm(kr[:,None] - k[None], axis=2).min(axis=1).max() < 1e-6
    return a, b, perm, err

def obedience(label, report, kfile, e1, e2):
    ops, coef, th, dE, _ = parse(report)
    k = np.load(kfile)
    N = len(th)
    print(f"{label}   N={N}")
    print(f"  {'χ':>3} {'θ 的作用':>26} {'是样点置换':>11} {'RMS 残差 (eV)':>16} {'去掉 l=0':>13}")
    rows = []
    for chi, R in ops:
        a, b, perm, _ = plane_action(R, k, e1, e2)
        kr = k @ R.T
        j  = np.linalg.norm(kr[:,None]-k[None], axis=2).argmin(axis=1)
        if not perm:
            print(f"  {chi:>+3d} {'θ -> %+d·θ %+.4f'%(a,b):>26} {'否 ★':>11}  —— 圆周上不是样点置换，无法逐点检验")
            continue
        res  = dE[j] - chi*dE
        rms  = float(np.sqrt((res**2).mean()))
        rms0 = float(np.sqrt(((res-res.mean())**2).mean()))
        print(f"  {chi:>+3d} {'θ -> %+d·θ %+.4f'%(a,b):>26} {'是':>11} {rms:>16.3e} {rms0:>13.3e}")
        rows.append((chi, rms0))
    return rows

rows_yz = obedience("k_y k_z 大圆", YZ, YZ_K, np.array([0,1.,0]), np.array([0,0,1.]))
print()
rows_xy = obedience("k_x k_y 圆周 @ k_z=0.25b₃", XY, XY_K, np.array([1.,0,0]), np.array([0,1.,0]))

k_y k_z 大圆   N=152
    χ                      θ 的作用       是样点置换      RMS 残差 (eV)        去掉 l=0
   +1          θ -> +1·θ +0.0000           是        0.000e+00     0.000e+00
   +1          θ -> +1·θ -3.1416           是        2.758e-14     2.758e-14
   -1          θ -> -1·θ -3.1416           是        4.860e-06     1.089e-06
   -1          θ -> -1·θ +0.0000           是        4.860e-06     1.089e-06

k_x k_y 圆周 @ k_z=0.25b₃   N=152
    χ                      θ 的作用       是样点置换      RMS 残差 (eV)        去掉 l=0
   +1          θ -> +1·θ +0.0000           是        0.000e+00     0.000e+00
   +1          θ -> +1·θ +2.0944         否 ★  —— 圆周上不是样点置换，无法逐点检验
   +1          θ -> +1·θ -2.0944         否 ★  —— 圆周上不是样点置换，无法逐点检验
   +1          θ -> -1·θ +3.1416           是        7.235e-08     7.235e-08
   +1          θ -> -1·θ +1.0472         否 ★  —— 圆周上不是样点置换，无法逐点检验
   +1          θ -> -1·θ -1.0472         否 ★  —— 圆周上不是样点置换，无法逐点检验
   -1          θ -> -1·θ -0.0000           是        6.147e-06     1.604e-06


**两条圆周给出同一个结构：**

| 圆周 | χ=+1（样点置换） | χ=−1 | 比 |
|---|---|---|---|
| k_y k_z | 2.8×10⁻¹⁴ eV | 1.09 μeV | 4×10⁷ |
| k_x k_y | 0.072 μeV | 1.60 μeV | 22 |

物理上这是两件不同的事：

- **χ=+1** 把 ε↑(k) 和 ε↑(Rk) 联系起来——同一个哈密顿量、同一个自旋块，
  QE 的对角化按构造就满足它。剩下的只是 k 点摆放的舍入误差。
- **χ=−1** 把 ε↑(k) 和 ε↓(Rk) 联系起来——它要求**自洽自旋密度**尊重子格交换。
  这只在 SCF 收敛的精度内成立，这里是 ~1 μeV。

y-z 圆周上还有一个单独的 **l=0 刚性偏移 4.74 μeV**（两个自旋通道的平均不完全简并），
它不属于任何被报道的谐波，所以不污染 l≥1 的底。

**第一个教训：** 若某个谐波只被 χ=+1 的操作禁戒，它会给出机器零，
把噪声底报得低 10⁸ 倍。噪声底读在哪个通道上，不是无关紧要的。

## 4. 检验二：已发表的 23.6 μeV 噪声底，到底读自哪里？

上面逐操作的残差最大才 1.6 μeV，而已发表的底是 23.6 μeV——差了 15 倍。
说明底一定读自那些**不是样点置换**的操作所禁戒的谐波。把 90 个系数按"被什么禁戒"分组。

In [6]:
def block(l, a, b):
    """θ -> a·θ+b 时作用在 (a_l, b_l) 上的 2×2 矩阵。反射会把 cos 混进 sin，
       所以约束是对这个 2×2 块的，不是对单个基函数的。"""
    c, s = np.cos(l*b), np.sin(l*b)
    return np.array([[c, s], [-a*s, a*c]])

def classify(report, kfile, e1, e2, lmax):
    ops, coef, th, dE, _ = parse(report)
    k = np.load(kfile)
    acts = [(chi,) + plane_action(R, k, e1, e2)[:3] for chi, R in ops]
    g = {'允许': [], '被“可表示”的操作禁戒': [], '只被“不可表示”的操作禁戒': []}
    for l in range(1, lmax+1):
        for name, v in (('c', np.array([1.,0.])), ('s', np.array([0.,1.]))):
            bad_perm = bad_free = False
            for chi, a, b, perm in acts:
                # b 由有限精度的 k 点反推，带 ~5e-10 rad；比较的是 cos(l·b)，
                # 误差随 l·δb 线性增长，所以容差必须随 l 走（真违反是 O(1)）
                if not np.allclose(block(l,a,b) @ v, chi*v, atol=1e-6*max(1,l)):
                    if perm: bad_perm = True
                    else:    bad_free = True
            key = '允许' if not (bad_perm or bad_free) else (
                  '被“可表示”的操作禁戒' if bad_perm else '只被“不可表示”的操作禁戒')
            g[key].append((l, name, abs(coef.get((l,name),0.0))*1e6))
    return g

g_xy = classify(XY, XY_K, np.array([1.,0,0]), np.array([0,1.,0]), 45)
print("Cr₂SeAs  k_x k_y 圆周, 152 点, lmax=45\n")
print(f"  {'组':<30} {'n':>4} {'中位 (μeV)':>13} {'最大 (μeV)':>13}")
for key, v in g_xy.items():
    a = np.array([x[2] for x in v])
    ex = ', '.join(f"{l}{n}" for l,n,_ in v[:6])
    print(f"  {key:<30} {len(a):>4} {np.median(a):>13.4f} {a.max():>13.2f}   例: {ex} …")

Cr₂SeAs  k_x k_y 圆周, 152 点, lmax=45

  组                                 n      中位 (μeV)      最大 (μeV)
  允许                                8     3093.2311      64292.77   例: 3s, 9s, 15s, 21s, 27s, 33s …
  被“可表示”的操作禁戒                      67        0.0040          1.11   例: 1c, 2c, 2s, 3c, 4c, 4s …
  只被“不可表示”的操作禁戒                    15       24.7022         75.04   例: 1s, 5s, 7s, 11s, 13s, 17s …


**这就是答案。** 已发表的 23.6 μeV 正是第三组的中位数（24.70 μeV），
它完全读自那 15 个**只被 C3 旋转禁戒**的 sin 通道。而 120° 不是 360°/152 的整数倍，
**C3 恰恰是这个网格唯一表示不了的操作**。被网格能表示的操作所禁戒的那 67 个系数，
位于 0.0040 μeV——低 6000 倍。

所以噪声底测的不是"数值误差"，而是**C3 这一条约束的有限 k 混叠**。
这从机制上印证了 P3 当年的经验结论（56→152→304 点时底从 9 meV 降到 1.01 μeV）。

但它立刻给出一个尖锐的预言：**若 N 能被 6 整除，12 个操作全都变成样点置换，
底会塌掉三个数量级——而物理一点没变。** 152 = 8×19，既不被 3 也不被 6 整除；150 = 6×25 则可以。

## 5. 检验三：在真值已知的合成数据上验证机制

真实 DFT 数据没有"真值"可比。所以构造一个**完全对称、零噪声**的信号：
只含允许谐波 l ≡ 3 (mod 6)，一路延伸到远超 lmax，幅度按实测系数外推。
完美的拟合应当在每个禁戒 l 上给出 0，并精确复现低阶系数。偏离的部分就全是混叠。

In [7]:
MEASURED = {3:64292.8, 9:43446.7, 15:3603.3, 21:8661.7,
            27:1876.3, 33:2583.2, 39:1181.9, 45:985.3}       # μeV, 152 点实测

def synthetic(n, decay=-1.5, lmax=45, l_true_max=1200):
    amps = dict(MEASURED)
    for l in range(51, l_true_max, 6):
        amps[l] = 985.3*(l/45.0)**decay*(1 + 0.3*np.sin(l/7.0))
    th = 2*np.pi*np.arange(n)/n
    f  = sum(a*np.sin(l*th) for l, a in amps.items())
    D  = np.column_stack([np.ones(n)] + [g(l*th) for l in range(1,lmax+1) for g in (np.cos,np.sin)])
    c, *_ = np.linalg.lstsq(D, f, rcond=None)
    co = {(l,k): c[1+2*(l-1)+j] for l in range(1,lmax+1) for j,k in enumerate('cs')}
    floor = max(abs(v) for (l,k),v in co.items() if not (l%6==3 and k=='s'))
    err   = 100*max(abs(co[(l,'s')]-MEASURED[l])/MEASURED[l] for l in MEASURED)
    return floor, err

print(f"  {'N':>4} {'gcd(N,6)':>9} {'混叠阶 m':>9} {'报出的噪声底':>15} {'允许系数真实误差':>18}")
for n in (150, 156, 162, 147, 153, 159, 146, 152, 160, 151, 155, 157):
    fl, er = synthetic(n)
    print(f"  {n:>4} {gcd(n,6):>9} {6//gcd(n,6):>9} {fl:>12.2f} μeV {er:>16.3f} %")

     N  gcd(N,6)     混叠阶 m          报出的噪声底           允许系数真实误差
   150         6         1         0.00 μeV           26.014 %


   156         6         1         0.00 μeV           19.608 %


   162         6         1         0.00 μeV           15.031 %


   147         3         2       199.55 μeV            3.674 %


   153         3         2       160.00 μeV            5.304 %
   159         3         2        98.04 μeV            3.683 %
   146         2         3       316.39 μeV            2.200 %


   152         2         3       220.88 μeV            1.443 %
   160         2         3       216.77 μeV            1.570 %
   151         1         6       239.00 μeV            0.215 %


   155         1         6       239.47 μeV            0.346 %
   157         1         6       218.55 μeV            0.251 %


**机制。** 材料的劈裂在**每一阶**都只含允许谐波；Cr₂SeAs 的允许集是 l ≡ 3 (mod 6)，
是一个周期 p = 6 的格。lmax 以上那条尾巴分辨不出来，在 N 个样点上混叠到 l' = |l − mN|。
l' 是否仍然允许，取决于 mN ≡ 0 (mod p)，所以尾巴**首次触到信号**的混叠阶是

$$m = p \,/\, \gcd(N, p)$$

- **p | N 时 m = 1**：每个未分辨谐波最强的那个混叠像直接加到某个被报道的系数上，
  而**没有任何一个禁戒通道收到东西**。底读零，答案错。
- **gcd 小时**：首次混叠落进禁戒通道，在那里作为底可见；要到第 m 阶才污染系数，
  那时尾巴已经衰减掉了。

## 6. 这条规律稳不稳？扫全部 N 和多种尾巴衰减

In [8]:
import collections
by_order = collections.defaultdict(list)
for n in range(120, 200):
    by_order[6//gcd(n,6)].append(synthetic(n)[1])
print("  N = 120…199，按混叠阶分组的**真实误差**：")
print(f"    {'m':>3} {'n':>4} {'中位':>9} {'最大':>9}")
for m in sorted(by_order):
    v = np.array(by_order[m])
    print(f"    {m:>3} {len(v):>4} {np.median(v):>8.3f}% {v.max():>8.3f}%")

print("\n  判据 'N 被 6 整除 => 底恰为 0 且系数被污染' 的反例：")
bad = [n for n in range(120,200) if n%6==0 and not (synthetic(n)[0] < 1e-6 and synthetic(n)[1] > 1.0)]
print(f"    {bad if bad else '无 ✓'}")

print("\n  换五种尾巴衰减，'危险网格看起来更好、实际更差' 是否都成立：")
for decay in (-1.0,-1.5,-2.0,-2.5,-3.0):
    b, g_ = synthetic(150, decay=decay), synthetic(151, decay=decay)
    print(f"    l^{decay:<4}: N=150 底 {b[0]:7.2f}/误差 {b[1]:6.2f}%   "
          f"N=151 底 {g_[0]:7.2f}/误差 {g_[1]:5.2f}%   "
          f"{'✓' if (b[0]<g_[0] and b[1]>g_[1]) else '✗'}")

  N = 120…199，按混叠阶分组的**真实误差**：
      m    n        中位        最大
      1   14   18.083%   36.236%
      2   13    3.683%    6.316%
      3   26    1.398%    2.584%
      6   27    0.292%    1.042%

  判据 'N 被 6 整除 => 底恰为 0 且系数被污染' 的反例：


    无 ✓

  换五种尾巴衰减，'危险网格看起来更好、实际更差' 是否都成立：
    l^-1.0: N=150 底    0.00/误差  38.27%   N=151 底  370.95/误差  1.01%   ✓


    l^-1.5: N=150 底    0.00/误差  26.01%   N=151 底  239.00/误差  0.22%   ✓
    l^-2.0: N=150 底    0.00/误差  17.95%   N=151 底  153.46/误差  0.06%   ✓


    l^-2.5: N=150 底    0.00/误差  12.32%   N=151 底   98.15/误差  0.02%   ✓
    l^-3.0: N=150 底    0.00/误差   8.37%   N=151 底   62.62/误差  0.00%   ✓


## 7. 结论

| gcd(N,6) | 混叠阶 | **报出的噪声底** | **真实误差**（中位/最大） |
|---|---|---|---|
| 6 | 1 | **0 μeV** | **18.1 % / 36.2 %** |
| 3 | 2 | 160 μeV | 3.7 % / 6.3 % |
| 2 | 3 | 221 μeV | 1.4 % / 2.6 % |
| 1 | 6 | 239 μeV | **0.29 % / 1.04 %** |

单调，N=120…199 无反例，五种尾巴衰减都成立。

> **噪声底与真实误差单调反相关。底读得最漂亮的网格，精度比底最难看的网格差 60 倍。**

和已有的五个盲区比，这一个性质不同：那五个是**盲**（底注意不到误差），
这一个是**反指**（底在误差最大的时候读零）。

**操作上的建议：取 N 与允许集周期互素，绝不取它的倍数。**
本项目自己用的 152 是 gcd=2 的情形——不是灾难性的那种，但比互素的取法差约 5 倍，
而且当初是碰巧选的，不是有意选的。

## 8. 两条独立路径的交叉验证

In [9]:
from altharm import CircularBasis, allowed_l, spin_operations
txt  = (RUNS/'hw_yz/bands.in').read_text().splitlines()
i    = next(n for n,l in enumerate(txt) if 'CELL_PARAMETERS' in l)
cell = np.array([[float(x) for x in txt[i+1+j].split()] for j in range(3)])*7.040816482142
j    = next(n for n,l in enumerate(txt) if 'ATOMIC_POSITIONS' in l)
sym  = [ln.split()[0] for ln in txt[j+1:j+11]]
pos  = np.array([[float(x) for x in ln.split()[1:4]] for ln in txt[j+1:j+11]])
Z    = {'CrU':24,'CrD':24,'Se':34,'As':33}
mag  = [1.0 if s=='CrU' else (-1.0 if s=='CrD' else 0.0) for s in sym]
ops  = spin_operations(cell, pos, [Z[s] for s in sym], mag, symprec=1e-3)

for label, rep, kf, e1, e2, lmax in [("k_y k_z", YZ, YZ_K, (0,1,0), (0,0,1), 24),
                                     ("k_x k_y", XY, XY_K, (1,0,0), (0,1,0), 45)]:
    k = np.load(kf); dirs = k/np.linalg.norm(k, axis=1, keepdims=True)
    lib  = sorted(allowed_l(CircularBasis(lmax, e1=e1, e2=e2), ops, dirs))
    mine = sorted(l for l,_,_ in classify(rep, kf, np.array(e1,float), np.array(e2,float), lmax)['允许'])
    print(f"{label}  (lmax={lmax})")
    print(f"   altharm.symmetry  (spglib ← 结构)       : {lib}")
    print(f"   本 notebook 的分类器 (← harmwave 的报告) : {mine}")
    print(f"   {'✓ 一致' if lib==mine else '✗ 不一致'}\n")

k_y k_z  (lmax=24)
   altharm.symmetry  (spglib ← 结构)       : [2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24]
   本 notebook 的分类器 (← harmwave 的报告) : [2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24]
   ✓ 一致



k_x k_y  (lmax=45)
   altharm.symmetry  (spglib ← 结构)       : [3, 9, 15, 21, 27, 33, 39, 45]
   本 notebook 的分类器 (← harmwave 的报告) : [3, 9, 15, 21, 27, 33, 39, 45]
   ✓ 一致



两条路径共享的东西只有那块晶体：一条从结构经 spglib 求对称性，
另一条读 harmwave.x 自己写进报告里的操作矩阵。它们在两个平面上都给出同一个允许集。

## 9. 已经代码化的部分

In [10]:
from altharm.sampling import sampling_report, alias_order, allowed_period
print("允许集周期:", allowed_period([3,9,15,21,27,33,39,45]), "(k_x k_y)  /",
      allowed_period(range(2,25,2)), "(k_y k_z)\n")
for n in (150, 152, 151):
    r = sampling_report(n, [3,9,15,21,27,33,39,45])
    print(f"  N={n}  safe={r.safe}  order={r.order}\n    {r.message}\n")

允许集周期: 6 (k_x k_y)  / 2 (k_y k_z)

  N=150  safe=False  order=1
    n = 150 is a multiple of the allowed-set period p = 6. Every unresolved harmonic above lmax aliases straight onto an ALLOWED l at first order, so the forbidden-harmonic floor will read essentially zero while the reported coefficients carry the whole tail as error (~18% on Cr2SeAs). Use a count coprime to 6, e.g. 151.

  N=152  safe=False  order=3
    n = 152 shares the factor 2 with the allowed-set period p = 6: the tail reaches the allowed coefficients at alias order 3. Usable, but a count coprime to 6 (e.g. 151) is strictly better at the same cost.

  N=151  safe=True  order=6
    n = 151 is coprime to the allowed-set period p = 6; the tail reaches the allowed coefficients only at alias order 6, and its first alias is visible in the forbidden channels.



## 9.5 机制在真实本征值上的测量：同一个体系，两个 N

第 5–6 节的合成数据真值已知但**没有任何 SCF 噪声**；第 3–4 节是真实数据但只有一个取样数。
304 点那次运行把两者接上，而且不需要新的 DFT——它被用两次：

1. 拟合到它**自己的 Nyquist**（l=151），直接测出高阶尾巴 a_111 … a_147；
2. **隔点取样**成 152 点——同一批本征值上的一条 152 点圆周——只拟合到 lmax=45，读它的禁戒通道。

折叠规律带符号：152 点网格上，l = 152−l′ 时 sin(lθ_j) = −sin(l′θ_j)，
所以预言是 **b_l′ = −b_(152−l′)**，符号和大小一起。

> k304 的 degauss 是 0.0037（旧设置），与 sm24 的 0.0150 不同。
> 所以这个检验完全在 **304 这一次运行内部**做，不跨密度比较。

In [11]:
from altharm.qe import read_bands, spin_splitting

qb = read_bands(ROOT/'tests/data/qe_cr2seas_k304/data-file-schema.xml')
kk = qb.kpoints_tpiba
dd = spin_splitting(qb, 44)                    # band_index 是 0 基的 -> 带 44
tt = np.arctan2(kk[:,1], kk[:,0]) % (2*np.pi)
o  = np.argsort(tt); tt, dd = tt[o], dd[o]
print(f"N = {len(tt)}   半径 {np.linalg.norm(kk[:,:2],axis=1).mean():.6f}   k_z = {kk[:,2].mean():.6f}")
print(f"θ 均匀性: max|Δθ−2π/N| = {np.abs(np.diff(tt)-2*np.pi/len(tt)).max():.2e}")

def lsqfit(t, f, lmax):
    D = np.column_stack([np.ones(len(t))] + [g(l*t) for l in range(1,lmax+1) for g in (np.cos,np.sin)])
    c, *_ = np.linalg.lstsq(D, f, rcond=None)
    return {(l,kc): c[1+2*(l-1)+j] for l in range(1,lmax+1) for j,kc in enumerate('cs')}

hi = lsqfit(tt, dd, 151)           # 304 点 -> 直接分辨到 l=151
lo = lsqfit(tt[::2], dd[::2], 45)  # 隔点取样 -> 152 点, 只到 l=45
PAIRS = [(5,147),(11,141),(17,135),(23,129),(29,123),(35,117),(41,111)]

print(f"\n  {'l′':>4} {'折叠自':>7} {'b_l′ 实测(152点)':>20} {'−b_(152−l′) 预言':>20} {'相对差':>9}")
res = []
for lp, src in PAIRS:
    got, pred = lo[(lp,'s')]*1e6, -hi[(src,'s')]*1e6
    res.append(abs((got-pred)/pred))
    print(f"  {lp:>4} {src:>7} {got:>+17.3f} μeV {pred:>+17.3f} μeV {100*(got-pred)/abs(pred):>+8.2f}%")

signs_ok = all(np.sign(lo[(a,'s')]) == np.sign(-hi[(b,'s')]) for a,b in PAIRS)
print(f"\n  符号全部正确: {'✓' if signs_ok else '✗'}")
print(f"  残差随 l′ 单调缩小: {'✓' if res==sorted(res,reverse=True) else '✗'}"
      f"  (剩下的是第二个折叠伙伴 l=152+l′，304 点测不到)")

print("\n  对照——允许系数不该被尾巴动到 (两个 N 都是 gcd=2，都不是一阶混叠):")
print(f"  {'l':>4} {'152点':>14} {'304点':>14} {'相对差':>10}")
for l in (3,9,15,21,27,33,39,45):
    a, b = abs(lo[(l,'s')])*1e6, abs(hi[(l,'s')])*1e6
    print(f"  {l:>4} {a:>11.1f} μeV {b:>11.1f} μeV {100*(a-b)/b:>+9.3f}%")

N = 304   半径 0.288675   k_z = 0.055057
θ 均匀性: max|Δθ−2π/N| = 3.18e-10



    l′     折叠自        b_l′ 实测(152点)       −b_(152−l′) 预言       相对差
     5     147           -33.755 μeV           -32.617 μeV    -3.49%
    11     141           +40.339 μeV           +39.324 μeV    +2.58%
    17     135           -44.636 μeV           -43.724 μeV    -2.08%
    23     129           +54.666 μeV           +53.835 μeV    +1.54%
    29     123           -60.772 μeV           -60.059 μeV    -1.19%
    35     117           +74.249 μeV           +73.591 μeV    +0.89%
    41     111           -86.259 μeV           -85.661 μeV    -0.70%

  符号全部正确: ✓
  残差随 l′ 单调缩小: ✓  (剩下的是第二个折叠伙伴 l=152+l′，304 点测不到)

  对照——允许系数不该被尾巴动到 (两个 N 都是 gcd=2，都不是一阶混叠):
     l           152点           304点        相对差
     3     59171.2 μeV     59171.4 μeV    -0.000%
     9     38817.2 μeV     38817.0 μeV    +0.000%
    15      3161.8 μeV      3161.6 μeV    +0.006%
    21      8421.1 μeV      8420.9 μeV    +0.002%
    27      1486.0 μeV      1485.8 μeV    +0.013%
    33      2838.1 μeV      2838.0 μeV    +0

**所以禁戒谐波的"噪声底"根本不是噪声**——它是信号自己那条分辨不出来的尾巴折叠下来的像，
可以用对那条尾巴的独立测量**连符号一起**预言到百分之几。

这就是真实本征值上的"同一个体系、不同 N、实测"，不需要新的 DFT。
但它测的是**机制**，还不是**灾难情形**：304 的任何等间隔子集都不能被 3 整除，
所以 m=1 那一档（底读零、系数吸收尾巴）目前仍然只有合成证据。

`altharm/sampling.py`（新增）+ `tests/test_p14_sampling_aliasing.py`（14 个测试）。
整套 288 个测试通过（原 274 + 新 14）。

## 10. 还**没有**做的事

9.5 节已经在真实本征值上测到了机制本身。还没有测到的是**灾难情形**：
304 的等间隔子集都不能被 3 整除，所以 m=1 那一档仍然只有合成证据。

**DFT 确认还没跑。** 输入已经备在 `altharm_runs/nsweep/`（N=150 / 151 / 153，
复用 sm24 同一个密度，每个 N 独立 outdir）。预言是：

- **N=150**（gcd=6）：底塌到 ~0.004 μeV，而允许系数相对 152 的值移动 ~10%
- **N=151**（互素）：底与 152 同量级，系数最准
- **N=153**（gcd=3）：居中

在它跑完之前，第 4 节是关于 152 点那次运行的一个**观测**，第 5–6 节是关于一个**模型**的陈述，
两者都还不是"N=150 对 Cr₂SeAs 做了什么"的**测量**。